# 06 · Publish: Hugging Face Hub + results GIF

1. Push the three LoRA adapters (SFT, PPO, GRPO) with model cards.
2. Push the golden evaluation set as a dataset.
3. Make the animated results GIF for the LinkedIn post (`docs/linkedin_post.md`).

Run on Colab **in the same runtime** as notebooks 01–05 (it needs `outputs/*-adapter` and `results/*.json`).
Before running: set `HF_USER` in `src/medrl/config.py`.

## 1. Setup and login

In [ ]:
# Setup (Colab): mount Drive, get the code, keep outputs/ and results/ in your Drive folder "RHLF"
import os, sys, glob
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    found = glob.glob("/content/drive/MyDrive/**/RHLF", recursive=True)
    DRIVE_DIR = found[0] if found else "/content/drive/MyDrive/RHLF"
    # Code comes from RHLF/code in Drive (the GitHub repo is private)
    !rm -rf /content/medalign-rl && cp -r "{DRIVE_DIR}/code" /content/medalign-rl
    assert os.path.exists("/content/medalign-rl/src/medrl"), "Upload the code folder to RHLF/code first"
    %cd /content/medalign-rl
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao  # Colab's old torchao breaks PEFT; we don't need it
    for d in ("outputs", "results"):
        os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
        if not os.path.islink(d):
            !cp -rn {d}/. {DRIVE_DIR}/{d}/ 2>/dev/null; rm -rf {d}
            os.symlink(f"{DRIVE_DIR}/{d}", d)
    print("Saving to", DRIVE_DIR)
sys.path.insert(0, "src")

In [ ]:
import sys
sys.path.insert(0, "src")
from huggingface_hub import notebook_login
notebook_login()   # paste a token with *write* access

In [ ]:
from medrl.config import HF_USER, OUTPUTS_DIR, DATA_DIR, RESULTS_DIR
from medrl.publish import load_all_scores, model_card, push_adapter, push_golden, make_gif
assert HF_USER != "your-hf-username", "Set HF_USER in src/medrl/config.py first"
scores = load_all_scores()
scores

## 2. Preview one model card

In [ ]:
from IPython.display import Markdown
Markdown(model_card("grpo", scores))

## 3. Push adapters and the golden set

In [ ]:
for name in ["sft", "ppo", "grpo"]:
    d = OUTPUTS_DIR / f"{name}-adapter"
    print(name, push_adapter(name, d, scores) if d.exists() else "missing " + str(d))
print(push_golden(DATA_DIR / "golden.jsonl"))

## 4. Results GIF

In [ ]:
from IPython.display import Image
gif = RESULTS_DIR / "06_results.gif"
make_gif(scores, gif)
Image(filename=str(gif))

## 5. Next

- Download `results/06_results.gif` and commit the `results/` files to GitHub.
- Fill the numbers into `docs/linkedin_post.md` and post with the GIF.